In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Pole / Obstacle Detection — Training Pipeline

**Steps**
1. Install dependencies
2. Configure paths
3. Annotate images ← draw bounding boxes here
4. Sanity-check labels
5. Split dataset (train 80% / val 15% / test 5%)
6. Train YOLOv8n
7. Evaluate trained model
8. Dump model pipeline as `.pkl`
9. Export for Jetson Nano

## Step 0 — Install dependencies

In [11]:
import sys
!{sys.executable} -m pip install ultralytics opencv-python numpy certifi ipywidgets ipycanvas pillow imblearn --quiet

## Step 1 — Configure paths

In [12]:
import os, sys, subprocess
from pathlib import Path

# ── Change this if running on Colab / different machine ──────────────────────
REPO_ROOT = Path("/content/drive/MyDrive/Jetson-NanoTesting")
# On Colab: REPO_ROOT = Path("/content/drive/MyDrive/JetsonNano-Testing")
# ─────────────────────────────────────────────────────────────────────────────

TRAINING_DIR  = REPO_ROOT / "DataTraining"
TOOLS_DIR     = TRAINING_DIR / "tools"
DATA_DIR      = TRAINING_DIR / "data"
RAW_IMAGES    = DATA_DIR / "raw_images"
RAW_LABELS    = DATA_DIR / "raw_labels"
DATASET_DIR   = DATA_DIR / "dataset"
RUNS_DIR      = TRAINING_DIR / "runs"
CLASSES_FILE  = TRAINING_DIR / "classes.txt"
BASE_MODEL    = REPO_ROOT / "yolov8n.pt"

PYTHON = sys.executable

# Load class names (used by annotator + sanity check)
class_names = [l.strip().lstrip('\ufeff') for l in CLASSES_FILE.read_text().splitlines()
               if l.strip() and not l.startswith("#")]

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"Python      : {PYTHON}")
print(f"Classes     : {dict(enumerate(class_names))}")
print(f"Base model  : {BASE_MODEL} (exists={BASE_MODEL.exists()})")
print(f"Raw images  : {len(list(RAW_IMAGES.glob('*.*')))} files")
print(f"Raw labels  : {len(list(RAW_LABELS.glob('*.txt')))} files")

REPO_ROOT   : /content/drive/MyDrive/Jetson-NanoTesting
Python      : /usr/bin/python3
Classes     : {0: 'rock', 1: 'pole', 2: 'crater', 3: 'unknown_obstacle'}
Base model  : /content/drive/MyDrive/Jetson-NanoTesting/yolov8n.pt (exists=True)
Raw images  : 41 files
Raw labels  : 43 files


In [13]:
import os, sys, subprocess
from pathlib import Path

# ── Change this if running on Colab / different machine ──────────────────────
# Original local path:
# REPO_ROOT = Path("/Users/satyasai/Documents/doc/JetsonNano-Testing")
# On Colab, assuming 'JetsonNano-Testing' is in your MyDrive:
REPO_ROOT = Path("/content/drive/MyDrive/Jetson-NanoTesting")
# ─────────────────────────────────────────────────────────────────────────────

TRAINING_DIR  = REPO_ROOT / "DataTraining"
TOOLS_DIR     = TRAINING_DIR / "tools"
DATA_DIR      = TRAINING_DIR / "data"
RAW_IMAGES    = DATA_DIR / "raw_images"
RAW_LABELS    = DATA_DIR / "raw_labels"
DATASET_DIR   = DATA_DIR / "dataset"
RUNS_DIR      = TRAINING_DIR / "runs"
CLASSES_FILE  = TRAINING_DIR / "classes.txt"
BASE_MODEL    = REPO_ROOT / "yolov8n.pt"

PYTHON = sys.executable

# Load class names (used by annotator + sanity check)
class_names = [l.strip() for l in CLASSES_FILE.read_text().splitlines()
               if l.strip() and not l.startswith("#")]

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"Python      : {PYTHON}")
print(f"Classes     : {dict(enumerate(class_names))}")
print(f"Base model  : {BASE_MODEL} (exists={BASE_MODEL.exists()})")
print(f"Raw images  : {len(list(RAW_IMAGES.glob('*.*')))} files")
print(f"Raw labels  : {len(list(RAW_LABELS.glob('*.txt')))} files")

REPO_ROOT   : /content/drive/MyDrive/Jetson-NanoTesting
Python      : /usr/bin/python3
Classes     : {0: '\ufeffrock', 1: 'pole', 2: 'crater', 3: 'unknown_obstacle'}
Base model  : /content/drive/MyDrive/Jetson-NanoTesting/yolov8n.pt (exists=True)
Raw images  : 41 files
Raw labels  : 43 files


## Step 2 — Annotate images

**How to use:**
- Select a class from the dropdown
- **Click once** on the image to set the top-left corner (a dot appears)
- **Click again** to set the bottom-right corner → box is drawn automatically
- `Undo` removes the last box, `Clear` removes all boxes
- `Save` writes labels to disk, `← Prev` / `Next →` auto-save and navigate

> Works in VS Code, JupyterLab, and Colab.

In [14]:
from ipycanvas import Canvas, hold_canvas
import ipywidgets as w
from IPython.display import display
from PIL import Image as PILImage
import numpy as np

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
BOX_COLORS = {0: '#00ff00', 1: '#ff4444', 2: '#ffff00', 3: '#00aaff'}

image_files = sorted([p for p in RAW_IMAGES.iterdir()
                      if p.suffix.lower() in IMAGE_EXTS])
assert image_files, f"No images found in {RAW_IMAGES}"
RAW_LABELS.mkdir(parents=True, exist_ok=True)

CANVAS_W, CANVAS_H = 900, 580

# ── Label I/O ─────────────────────────────────────────────────────────────────
def load_labels(img_path):
    lbl = RAW_LABELS / (img_path.stem + '.txt')
    boxes = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                boxes.append([int(parts[0])] + [float(x) for x in parts[1:]])
    return boxes

def save_labels(img_path, boxes):
    lbl = RAW_LABELS / (img_path.stem + '.txt')
    lbl.write_text('\n'.join(
        f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}" for b in boxes
    ))

# ── State ──────────────────────────────────────────────────────────────────────
state = {'idx': 0, 'boxes': [], 'first_click': None, 'img_w': 1, 'img_h': 1}
state['boxes'] = load_labels(image_files[0])

# ── Canvas setup ──────────────────────────────────────────────────────────────
canvas = Canvas(width=CANVAS_W, height=CANVAS_H)

def draw():
    img_path = image_files[state['idx']]
    pil_img  = PILImage.open(img_path).convert('RGB')
    iw, ih   = pil_img.size
    state['img_w'], state['img_h'] = iw, ih

    # Scale image to fit canvas
    scale    = min(CANVAS_W / iw, CANVAS_H / ih)
    dw, dh   = int(iw * scale), int(ih * scale)
    ox, oy   = (CANVAS_W - dw) // 2, (CANVAS_H - dh) // 2
    state['scale'] = scale
    state['ox'], state['oy'] = ox, oy

    pil_img  = pil_img.resize((dw, dh), PILImage.LANCZOS)
    img_arr  = np.array(pil_img)

    with hold_canvas(canvas):
        canvas.clear()
        canvas.put_image_data(img_arr, ox, oy)

        # Draw saved boxes
        for b in state['boxes']:
            cls, cx, cy, bw, bh = b
            # YOLO → canvas pixels
            x1c = ox + (cx - bw/2) * iw * scale
            y1c = oy + (cy - bh/2) * ih * scale
            pwc = bw * iw * scale
            phc = bh * ih * scale
            color = BOX_COLORS.get(cls, '#ffffff')
            canvas.stroke_style = color
            canvas.line_width   = 2
            canvas.stroke_rect(x1c, y1c, pwc, phc)
            name = class_names[cls] if cls < len(class_names) else str(cls)
            canvas.fill_style = 'rgba(0,0,0,0.6)'
            canvas.fill_rect(x1c, y1c - 16, len(name) * 7 + 6, 16)
            canvas.fill_style = color
            canvas.font = 'bold 12px sans-serif'
            canvas.fill_text(name, x1c + 3, y1c - 3)

        # Draw first-click crosshair
        if state['first_click']:
            fx, fy = state['first_click']
            canvas.stroke_style = 'white'
            canvas.line_width   = 1.5
            canvas.stroke_rect(fx - 5, fy - 5, 10, 10)

        # Status text
        msg = (f"Image {state['idx']+1}/{len(image_files)} — "
               f"{len(state['boxes'])} box(es) | "
               + ("click corner 1" if state['first_click'] is None else "click corner 2"))
        canvas.fill_style = 'rgba(0,0,0,0.55)'
        canvas.fill_rect(0, 0, CANVAS_W, 20)
        canvas.fill_style = 'white'
        canvas.font = '11px sans-serif'
        canvas.fill_text(msg, 6, 14)

# ── Mouse click ────────────────────────────────────────────────────────────────
def on_mouse_down(x, y):
    ox, oy = state['ox'], state['oy']
    scale  = state['scale']
    iw, ih = state['img_w'], state['img_h']

    # Ignore clicks outside image area
    if not (ox <= x <= ox + iw*scale and oy <= y <= oy + ih*scale):
        return

    if state['first_click'] is None:
        state['first_click'] = (x, y)
        draw()
    else:
        x1c, y1c = state['first_click']
        state['first_click'] = None
        lx, rx = min(x1c, x), max(x1c, x)
        ty, by = min(y1c, y), max(y1c, y)
        if abs(rx - lx) > 5 and abs(by - ty) > 5:
            # Canvas pixels → YOLO normalised
            cx = ((lx + rx) / 2 - ox) / (iw * scale)
            cy = ((ty + by) / 2 - oy) / (ih * scale)
            bw = (rx - lx) / (iw * scale)
            bh = (by - ty) / (ih * scale)
            state['boxes'].append([class_dd.index, cx, cy, bw, bh])
        draw()

canvas.on_mouse_down(on_mouse_down)

# ── Button callbacks ───────────────────────────────────────────────────────────
def go_prev(_):
    save_labels(image_files[state['idx']], state['boxes'])
    state['idx'] = max(0, state['idx'] - 1)
    state['boxes'] = load_labels(image_files[state['idx']])
    state['first_click'] = None
    draw()

def go_next(_):
    save_labels(image_files[state['idx']], state['boxes'])
    state['idx'] = min(len(image_files) - 1, state['idx'] + 1)
    state['boxes'] = load_labels(image_files[state['idx']])
    state['first_click'] = None
    draw()

def do_save(_):
    save_labels(image_files[state['idx']], state['boxes'])
    status_lbl.value = f"Saved {len(state['boxes'])} box(es)"

def do_undo(_):
    state['first_click'] = None
    if state['boxes']:
        state['boxes'].pop()
    draw()

def do_clear(_):
    state['first_click'] = None
    state['boxes'] = []
    draw()

# ── Widgets ────────────────────────────────────────────────────────────────────
class_dd   = w.Dropdown(options=[(n, i) for i, n in enumerate(class_names)],
                         description='Class:', layout=w.Layout(width='160px'))
btn_prev   = w.Button(description='← Prev',  layout=w.Layout(width='80px'))
btn_next   = w.Button(description='Next →',  layout=w.Layout(width='80px'))
btn_save   = w.Button(description='Save',    button_style='success', layout=w.Layout(width='70px'))
btn_undo   = w.Button(description='Undo',    button_style='warning', layout=w.Layout(width='70px'))
btn_clear  = w.Button(description='Clear',   button_style='danger',  layout=w.Layout(width='70px'))
status_lbl = w.Label(value='')

btn_prev.on_click(go_prev)
btn_next.on_click(go_next)
btn_save.on_click(do_save)
btn_undo.on_click(do_undo)
btn_clear.on_click(do_clear)

toolbar = w.HBox([btn_prev, btn_next, class_dd, btn_undo, btn_save, btn_clear, status_lbl])
display(toolbar, canvas)
draw()

Canvas(height=580, width=900)

In [15]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [16]:
from google.colab import output
output.disable_custom_widget_manager()

## Step 3 — Sanity-check labels

In [17]:
from collections import Counter

counts: Counter = Counter()
bad_files = []
for lbl in sorted(RAW_LABELS.glob('*.txt')):
    for line in lbl.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            bad_files.append((lbl.name, line))
            continue
        try:
            counts[int(parts[0])] += 1
        except ValueError:
            bad_files.append((lbl.name, line))

print('Box counts per class:')
for cls_id, n in sorted(counts.items()):
    name = class_names[cls_id] if cls_id < len(class_names) else f'UNKNOWN({cls_id})'
    bar  = '█' * min(n, 50)
    print(f'  {cls_id} {name:<12} {bar} {n}')

if bad_files:
    print(f'\n⚠  {len(bad_files)} malformed lines:')
    for f, l in bad_files[:5]:
        print(f'   {f}: {l!r}')
else:
    print('\nAll label lines look valid.')

Box counts per class:
  0 ﻿rock        ██████████████████████████████████████████████████ 168
  2 crater       ████████████████████████████████████████████ 44

All label lines look valid.


## Step 4 — Split dataset

In [18]:
for cache in DATASET_DIR.glob('labels/*.cache'):
    cache.unlink()
    print(f'Deleted cache: {cache.name}')

result = subprocess.run(
    [
        PYTHON, str(TOOLS_DIR / 'split_dataset.py'),
        '--images-dir', str(RAW_IMAGES),
        '--labels-dir', str(RAW_LABELS),
        '--output-dir', str(DATASET_DIR),
        '--classes',    str(CLASSES_FILE),
        '--ratios',     '0.80,0.15,0.05',
        '--seed',       '42',
    ],
    cwd=str(REPO_ROOT), text=True, capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

for split in ('train', 'val', 'test'):
    n = len(list((DATASET_DIR / 'images' / split).glob('*.*')))
    print(f'  {split}: {n} images')

[DONE] Dataset split complete
  Total samples: 40
  Train: 32
  Val:   6
  Test:  2
  Dataset YAML: /content/drive/MyDrive/Jetson-NanoTesting/DataTraining/data/dataset/dataset.yaml

  train: 39 images
  val: 12 images
  test: 3 images


In [19]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import numpy as np # Needed for dummy data

# --- Explanation for X_train and y_train in an object detection context ---
# In an object detection pipeline like YOLO, 'X_train' and 'y_train' as used
# with SMOTE (feature vectors and corresponding single class labels for each sample)
# are not directly generated by the `split_dataset.py` script.
#
# If you intend to use SMOTE to balance the number of bounding boxes per class,
# you would typically need to:
# 1. Parse all training label files (e.g., from DATASET_DIR / 'labels' / 'train').
# 2. For each bounding box, extract its class ID and potentially its normalized
#    coordinates (cx, cy, w, h) as features.
# 3. 'y_train' would be the list of class IDs for all bounding boxes.
# 4. 'X_train' would be a 2D array where each row represents a bounding box
#    and columns are its features (e.g., [cx, cy, w, h]).
#
# As a demonstration, here are placeholders for X_train and y_train.
# You would replace these with your actual extracted bounding box data.
# -------------------------------------------------------------------------

# Placeholder: Create some dummy data for demonstration purposes.
# In a real scenario, you would populate X_train and y_train by parsing your
# training annotation files (e.g., from `DATASET_DIR / 'labels' / 'train'`).
# For instance, if you have 100 bounding boxes with 4 features each (cx, cy, w, h).
X_train = np.random.rand(100, 4) # 100 samples, 4 features (e.g., normalized bbox coords)
# Create an imbalanced y_train with 2 classes (e.g., class 0 'rock', class 1 'pole')
y_train = np.array([0]*80 + [1]*20) # 80 samples of class 0, 20 of class 1
np.random.shuffle(y_train) # Shuffle to mix the classes

# Instantiate SMOTE
sm = SMOTE(random_state=42)

# Apply SMOTE to resample the training data
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

print(f"Before SMOTE: {Counter(y_train)}")
print(f"After SMOTE: {Counter(y_resampled)}")

# Note: After applying SMOTE, you would need to consider how to integrate
# these new synthetic samples (bounding box features and class IDs) back
# into your object detection training process. This is not a straightforward
# step for most object detection frameworks like YOLO, which typically use
# data augmentation or class weights for imbalance.

Before SMOTE: Counter({np.int64(0): 80, np.int64(1): 20})
After SMOTE: Counter({np.int64(0): 80, np.int64(1): 80})


## Step 5 — Train YOLOv8n

In [20]:
import sys
from pathlib import Path
import subprocess

# --- Path Configuration (copied from config cell and adapted for Colab) ---
# Assuming you have mounted Google Drive and "JetsonNano-Testing" is in MyDrive
REPO_ROOT = Path("/content/drive/MyDrive/Jetson-NanoTesting")
# If running locally, you might need to change this back to your local path:
# REPO_ROOT = Path("/Users/satyasai/Documents/doc/JetsonNano-Testing")

TRAINING_DIR  = REPO_ROOT / "DataTraining"
TOOLS_DIR     = TRAINING_DIR / "tools"
DATA_DIR      = TRAINING_DIR / "data"
RAW_IMAGES    = DATA_DIR / "raw_images" # Not directly used in train, but good to have if needed later
RAW_LABELS    = DATA_DIR / "raw_labels" # Not directly used in train
DATASET_DIR   = DATA_DIR / "dataset"
RUNS_DIR      = TRAINING_DIR / "runs"
CLASSES_FILE  = TRAINING_DIR / "classes.txt" # Not directly used in train
BASE_MODEL    = REPO_ROOT / "yolov8n.pt"

PYTHON = sys.executable
# --- End Path Configuration ---

TRAIN_CONFIG = dict(
    data    = str(DATASET_DIR / 'dataset.yaml'),
    model   = str(BASE_MODEL),
    epochs  = '100',
    imgsz   = '640',
    batch   = '16',
    device  = 'cuda',
    workers = '0',
    # Removed unrecognized arguments: patience, mosaic, mixup, degrees, scale, fliplr
)

cmd = [PYTHON, str(TOOLS_DIR / 'train_yolo.py')]
for k, v in TRAIN_CONFIG.items():
    cmd += [f'--{k}', v]

print('Command:\n ', ' '.join(cmd), '\n')

proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
best_pt_path = None
save_dir_from_output = None

for line in proc.stdout:
    print(line, end='')
    if line.strip().startswith('Save dir: '):
        save_dir_from_output = line.strip().replace('Save dir: ', '')

proc.wait()

if save_dir_from_output:
    best_pt_path = str(Path(save_dir_from_output) / 'weights' / 'best.pt')
else:
    # Fallback to glob if 'Save dir:' line was not found, though it should be.
    # Search within REPO_ROOT / 'runs' to capture general ultralytics output structure
    hits = sorted(Path(REPO_ROOT / 'runs').glob('**/best.pt'),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        best_pt_path = str(hits[0])

print(f'\nBest model: {best_pt_path}')

Command:
  /usr/bin/python3 /content/drive/MyDrive/Jetson-NanoTesting/DataTraining/tools/train_yolo.py --data /content/drive/MyDrive/Jetson-NanoTesting/DataTraining/data/dataset/dataset.yaml --model /content/drive/MyDrive/Jetson-NanoTesting/yolov8n.pt --epochs 100 --imgsz 640 --batch 16 --device cuda --workers 0 

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0,

## Step 6 — Evaluate trained model

In [21]:
EVAL_SAVE_DIR = RUNS_DIR / 'eval_results'

# Ensure best_pt_path is explicitly a string and not None
# This makes the cell more robust if the 'train' cell wasn't run immediately before.
if 'best_pt_path' not in locals() or best_pt_path is None:
    print("Warning: best_pt_path is not defined or is None. Attempting to find it via glob fallback.")
    # Search within REPO_ROOT / 'runs' to capture general ultralytics output structure
    hits = sorted(Path(REPO_ROOT / 'runs').glob('**/best.pt'),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        best_pt_path = str(hits[0])
        print(f"Found best model path: {best_pt_path}")
    else:
        print("Error: Could not find best.pt. Please ensure the training step has been completed successfully.")
        # Raise an error or exit, as evaluation cannot proceed without a model
        raise FileNotFoundError("best.pt model not found. Run the training cell first.")

# Convert to string explicitly to be safe
model_path_str = str(best_pt_path)

result = subprocess.run(
    [
        PYTHON, str(TOOLS_DIR / 'eval_poles.py'),
        '--model',      model_path_str,
        '--images-dir', str(DATASET_DIR / 'images' / 'test'),
        '--labels-dir', str(DATASET_DIR / 'labels' / 'test'),
        '--conf',       '0.25',
        '--save-dir',   str(EVAL_SAVE_DIR),
    ],
    cwd=str(REPO_ROOT), text=True, capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

# Parse the summary table — format: Class  Precision  Recall  F1
eval_results = {}
parsing_data_rows = False
for line in result.stdout.splitlines():
    line = line.strip()
    if line.startswith('Class') and 'Precision' in line and 'Recall' in line and 'F1' in line:
        parsing_data_rows = True
        continue
    if parsing_data_rows and (not line or line.startswith('[DONE]')):
        parsing_data_rows = False
        break
    if parsing_data_rows and not line.startswith('---'):
        parts = line.split()
        if len(parts) >= 4:
            try:
                eval_results[parts[0]] = {
                    'precision': float(parts[-3]),
                    'recall':    float(parts[-2]),
                    'f1':        float(parts[-1]),
                }
            except ValueError:
                pass

print('\nParsed eval results:', eval_results)


Evaluation Results
Class       Precision    Recall        F1
-------------------------------------------
rock         0.579        0.850        0.689       
obstacle     0.528        0.820        0.642       
pole         0.619        0.882        0.727       
berm_marker  0.897        0.705        0.789       
hole_rgb     0.924        0.740        0.822       
unknown_obstacle 0.753        0.863        0.804       

[DONE] Evaluation complete


Parsed eval results: {'rock': {'precision': 0.579, 'recall': 0.85, 'f1': 0.689}, 'obstacle': {'precision': 0.528, 'recall': 0.82, 'f1': 0.642}, 'pole': {'precision': 0.619, 'recall': 0.882, 'f1': 0.727}, 'berm_marker': {'precision': 0.897, 'recall': 0.705, 'f1': 0.789}, 'hole_rgb': {'precision': 0.924, 'recall': 0.74, 'f1': 0.822}, 'unknown_obstacle': {'precision': 0.753, 'recall': 0.863, 'f1': 0.804}}


## Step 7 — Save model pipeline as `.pkl`

In [22]:
import pickle, datetime
from dataclasses import dataclass, field
from typing import Optional, Dict, Any


@dataclass
class ModelPipeline:
    """Portable snapshot of a trained model + all metadata needed to reproduce or deploy it."""
    model_path:      str
    class_names:     Dict[int, str]
    training_config: Dict[str, Any]
    eval_results:    Optional[Dict[str, Dict[str, float]]] = None
    created_at:      str = field(default_factory=lambda: datetime.datetime.now().isoformat())

    def save(self, path: str) -> None:
        with open(path, 'wb') as f:
            pickle.dump(self, f)
        print(f'Pipeline saved → {path}')

    @classmethod
    def load(cls, path: str) -> 'ModelPipeline':
        with open(path, 'rb') as f:
            return pickle.load(f)

    def export(self, output_format: str = 'onnx') -> str:
        from ultralytics import YOLO
        exported = YOLO(self.model_path).export(format=output_format)
        print(f'Exported → {exported}')
        return str(exported)

    def summary(self) -> None:
        print(f'Model      : {self.model_path}')
        print(f'Created    : {self.created_at}')
        print(f'Classes    : {self.class_names}')
        print(f'Train cfg  : {self.training_config}')
        if self.eval_results:
            print('Eval results:')
            for cls_name, m in self.eval_results.items():
                print(f'  {cls_name:<12} P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}')
        else:
            print('Eval results: not yet run')


pipeline = ModelPipeline(
    model_path      = best_pt_path,
    class_names     = dict(enumerate(class_names)),
    training_config = TRAIN_CONFIG,
    eval_results    = eval_results if eval_results else None,
)
pipeline.summary()

PKL_PATH = str(RUNS_DIR / 'pipeline.pkl')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
pipeline.save(PKL_PATH)

Model      : /content/drive/MyDrive/Jetson-NanoTesting/runs/detect/DataTraining/runs/pole_rock_train-12/weights/best.pt
Created    : 2026-04-24T23:05:04.493926
Classes    : {0: '\ufeffrock', 1: 'pole', 2: 'crater', 3: 'unknown_obstacle'}
Train cfg  : {'data': '/content/drive/MyDrive/Jetson-NanoTesting/DataTraining/data/dataset/dataset.yaml', 'model': '/content/drive/MyDrive/Jetson-NanoTesting/yolov8n.pt', 'epochs': '100', 'imgsz': '640', 'batch': '16', 'device': 'cuda', 'workers': '0'}
Eval results:
  rock         P=0.579  R=0.850  F1=0.689
  obstacle     P=0.528  R=0.820  F1=0.642
  pole         P=0.619  R=0.882  F1=0.727
  berm_marker  P=0.897  R=0.705  F1=0.789
  hole_rgb     P=0.924  R=0.740  F1=0.822
  unknown_obstacle P=0.753  R=0.863  F1=0.804
Pipeline saved → /content/drive/MyDrive/Jetson-NanoTesting/DataTraining/runs/pipeline.pkl


### Load pipeline back (verify round-trip)

In [23]:
loaded = ModelPipeline.load(PKL_PATH)
loaded.summary()

Model      : /content/drive/MyDrive/Jetson-NanoTesting/runs/detect/DataTraining/runs/pole_rock_train-12/weights/best.pt
Created    : 2026-04-24T23:05:04.493926
Classes    : {0: '\ufeffrock', 1: 'pole', 2: 'crater', 3: 'unknown_obstacle'}
Train cfg  : {'data': '/content/drive/MyDrive/Jetson-NanoTesting/DataTraining/data/dataset/dataset.yaml', 'model': '/content/drive/MyDrive/Jetson-NanoTesting/yolov8n.pt', 'epochs': '100', 'imgsz': '640', 'batch': '16', 'device': 'cuda', 'workers': '0'}
Eval results:
  rock         P=0.579  R=0.850  F1=0.689
  obstacle     P=0.528  R=0.820  F1=0.642
  pole         P=0.619  R=0.882  F1=0.727
  berm_marker  P=0.897  R=0.705  F1=0.789
  hole_rgb     P=0.924  R=0.740  F1=0.822
  unknown_obstacle P=0.753  R=0.863  F1=0.804


## Step 8 — Export for Jetson Nano

In [24]:
result = subprocess.run(
    [
        PYTHON, str(TOOLS_DIR / 'export_for_jetson.py'),
        '--model', best_pt_path,
    ],
    cwd=str(REPO_ROOT), text=True, capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

# Alternative: export via pipeline object
# exported_path = loaded.export(output_format='onnx')

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Jetson-NanoTesting/runs/detect/DataTraining/runs/pole_rock_train-12/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (6.0 MB)
requirements: Ultralytics requirement ['onnx>=1.12.0,<2.0.0'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 275ms
Prepared 1 package in 1.17s
Installed 1 package in 303ms
 + onnx==1.21.0

requirements: AutoUpdate success ✅ 2.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: export success ✅ 3.0s, saved as '/content/drive/MyD

---
## Quick-run: full pipeline in one cell
Runs steps 4 → 5 → 6 → 7 in sequence. Use this to retrain after adding new annotations.

In [25]:
import subprocess, sys, pickle, datetime
from pathlib import Path

# --- Path Configuration (adapted for Colab) ---
# Assuming you have mounted Google Drive and "JetsonNano-Testing" is in MyDrive
REPO_ROOT    = Path('/content/drive/MyDrive/Jetson-NanoTesting')
# If running locally, you might need to change this back to your local path:
# REPO_ROOT    = Path('/Users/satyasai/Documents/doc/JetsonNano-Testing')
# --- End Path Configuration ---

TRAINING_DIR = REPO_ROOT / 'DataTraining'
TOOLS_DIR    = TRAINING_DIR / 'tools'
DATA_DIR     = TRAINING_DIR / 'data'
DATASET_DIR  = DATA_DIR / 'dataset'
RUNS_DIR     = TRAINING_DIR / 'runs'
CLASSES_FILE = TRAINING_DIR / 'classes.txt'
BASE_MODEL   = REPO_ROOT / 'yolov8n.pt'
PYTHON       = sys.executable

class_names = [l.strip().lstrip('\ufeff') for l in CLASSES_FILE.read_text().splitlines()
               if l.strip() and not l.startswith("#")]

def run(cmd):
    proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    out = []
    for line in proc.stdout:
        print(line, end='')
        out.append(line)
    proc.wait()
    return '\n'.join(out)

# Split
for c in DATASET_DIR.glob('labels/*.cache'): c.unlink()
print('=' * 60, 'SPLIT', '=' * 60, sep='\n')
run([PYTHON, str(TOOLS_DIR/'split_dataset.py'),
     '--images-dir', str(DATA_DIR/'raw_images'),
     '--labels-dir', str(DATA_DIR/'raw_labels'),
     '--output-dir', str(DATASET_DIR),
     '--classes',    str(CLASSES_FILE),
     '--ratios',     '0.80,0.15,0.05',
     '--seed',       '42'])

# Train
print('=' * 60, 'TRAIN', '=' * 60, sep='\n')
train_out = run([PYTHON, str(TOOLS_DIR/'train_yolo.py'),
     '--data',    str(DATASET_DIR/'dataset.yaml'),
     '--model',   str(BASE_MODEL),
     '--epochs',  '100',
     '--batch',   '16',
     '--device',  'cuda',
     '--workers', '0',
     # Removed unrecognized arguments: patience, mosaic, mixup, degrees, scale, fliplr
     ])

best_pt = None
save_dir_from_train = None

for line in train_out.splitlines():
    if line.strip().startswith('Save dir: '):
        save_dir_from_train = line.strip().replace('Save dir: ', '')
        break # Found the line, no need to parse further

if save_dir_from_train:
    best_pt = str(Path(save_dir_from_train) / 'weights' / 'best.pt')
else:
    # Fallback if 'Save dir:' was not found
    hits = sorted(Path(REPO_ROOT / 'runs').glob('**/best.pt'), # Search within REPO_ROOT/runs for robustness
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        best_pt = str(hits[0])

# Eval
print('=' * 60, 'EVAL', '=' * 60, sep='\n')
eval_out = run([PYTHON, str(TOOLS_DIR/'eval_poles.py'),
     '--model',      best_pt,
     '--images-dir', str(DATASET_DIR/'images/test'),
     '--labels-dir', str(DATASET_DIR/'labels/test'),
     '--conf',       '0.25',
     '--save-dir',   str(RUNS_DIR/'eval_results')])

eval_results: dict = {}
in_table = False
for line in eval_out.splitlines():
    if line.startswith('===='): in_table = not in_table; continue
    if in_table and not line.startswith('Class') and not line.startswith('---'):
        parts = line.split()
        if len(parts) >= 4: # Changed from >= 8 to >= 4
            try:
                eval_results[parts[0]] = {
                    'precision': float(parts[-3]),
                    'recall':    float(parts[-2]),
                    'f1':        float(parts[-1]),
                }
            except ValueError: pass

# Save pkl
from dataclasses import dataclass, field
from typing import Optional, Dict, Any

@dataclass
class ModelPipeline:
    model_path: str
    class_names: Dict[int, str]
    training_config: Dict[str, Any]
    eval_results: Optional[Dict] = None
    created_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    def save(self, path):
        with open(path, 'wb') as f: pickle.dump(self, f)
        print(f'Pipeline saved → {path}')
    @classmethod
    def load(cls, path):
        with open(path, 'rb') as f: return pickle.load(f)
    def export(self, fmt='onnx'):
        from ultralytics import YOLO
        return str(YOLO(self.model_path).export(format=fmt))
    def summary(self):
        print(f'Model: {self.model_path}\nClasses: {self.class_names}\nCreated: {self.created_at}')
        if self.eval_results:
            for k, v in self.eval_results.items():
                print(f'  {k}: P={v["precision"]:.3f} R={v["recall"]:.3f} F1={v["f1"]:.3f}')

pipeline = ModelPipeline(
    model_path=best_pt,
    class_names=dict(enumerate(class_names)),
    training_config={'epochs': 100, 'batch': 16, 'device': 'cuda', 'imgsz': 640},
    eval_results=eval_results or None,
)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
pipeline.save(str(RUNS_DIR / 'pipeline.pkl'))
pipeline.summary()

SPLIT
[DONE] Dataset split complete
  Total samples: 40
  Train: 32
  Val:   6
  Test:  2
  Dataset YAML: /content/drive/MyDrive/Jetson-NanoTesting/DataTraining/data/dataset/dataset.yaml
TRAIN
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Jetson-NanoTesting/DataTraining/data/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None

In [26]:
import subprocess, sys, pickle, datetime
from pathlib import Path

# --- Path Configuration (adapted for Colab) ---
# Original local path:
# REPO_ROOT    = Path('/Users/satyasai/Documents/doc/JetsonNano-Testing')
# On Colab, assuming 'JetsonNano-Testing' is in your MyDrive:
REPO_ROOT    = Path('/content/drive/MyDrive/JetsonNano-Testing')
# --- End Path Configuration ---

TRAINING_DIR = REPO_ROOT / 'DataTraining'
TOOLS_DIR    = TRAINING_DIR / 'tools'
DATA_DIR     = TRAINING_DIR / 'data'
DATASET_DIR  = DATA_DIR / 'dataset'
RUNS_DIR     = TRAINING_DIR / 'runs'
CLASSES_FILE = TRAINING_DIR / 'classes.txt'
BASE_MODEL   = REPO_ROOT / 'yolov8n.pt'
PYTHON       = sys.executable

class_names = [l.strip() for l in CLASSES_FILE.read_text().splitlines()
               if l.strip() and not l.startswith('#')]

def run(cmd):
    proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    out = []
    for line in proc.stdout:
        print(line, end='')
        out.append(line)
    proc.wait()
    return '\n'.join(out)

# Split
for c in DATASET_DIR.glob('labels/*.cache'): c.unlink()
print('=' * 60, 'SPLIT', '=' * 60, sep='\n')
run([PYTHON, str(TOOLS_DIR/'split_dataset.py'),
     '--images-dir', str(DATA_DIR/'raw_images'),
     '--labels-dir', str(DATA_DIR/'raw_labels'),
     '--output-dir', str(DATASET_DIR),
     '--classes',    str(CLASSES_FILE),
     '--ratios',     '0.80,0.15,0.05',
     '--seed',       '42'])

# Train
print('=' * 60, 'TRAIN', '=' * 60, sep='\n')
train_out = run([PYTHON, str(TOOLS_DIR/'train_yolo.py'),
     '--data',    str(DATASET_DIR/'dataset.yaml'),
     '--model',   str(BASE_MODEL),
     '--epochs',  '100',
     '--batch',   '16',
     '--device',  'cuda',
     '--workers', '0',
     '--patience','30',
     '--mosaic',  '1.0',
     '--mixup',   '0.1',
     '--degrees', '10.0',
     '--scale',   '0.5',
     '--fliplr',  '0.5'])

best_pt = None
for line in train_out.splitlines():
    for tok in line.split():
        if tok.endswith('best.pt'): best_pt = tok
if not best_pt:
    hits = sorted(RUNS_DIR.glob('**/best.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    if hits: best_pt = str(hits[0])

# Eval
print('=' * 60, 'EVAL', '=' * 60, sep='\n')
eval_out = run([PYTHON, str(TOOLS_DIR/'eval_poles.py'),
     '--model',      best_pt,
     '--images-dir', str(DATASET_DIR/'images/test'),
     '--labels-dir', str(DATASET_DIR/'labels/test'),
     '--conf',       '0.25',
     '--save-dir',   str(RUNS_DIR/'eval_results')])

eval_results: dict = {}
in_table = False
for line in eval_out.splitlines():
    if line.startswith('===='): in_table = not in_table; continue
    if in_table and not line.startswith('Class') and not line.startswith('---'):
        parts = line.split()
        if len(parts) >= 8:
            try:
                eval_results[parts[0]] = {
                    'precision': float(parts[-3]),
                    'recall':    float(parts[-2]),
                    'f1':        float(parts[-1]),
                }
            except ValueError: pass

# Save pkl
from dataclass import dataclass, field
from typing import Optional, Dict, Any

@dataclass
class ModelPipeline:
    model_path: str
    class_names: Dict[int, str]
    training_config: Dict[str, Any]
    eval_results: Optional[Dict] = None
    created_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    def save(self, path):
        with open(path, 'wb') as f: pickle.dump(self, f)
        print(f'Pipeline saved → {path}')
    @classmethod
    def load(cls, path):
        with open(path, 'rb') as f: return pickle.load(f)
    def export(self, fmt='onnx'):
        from ultralytics import YOLO
        return str(YOLO(self.model_path).export(format=fmt))
    def summary(self):
        print(f'Model: {self.model_path}\nClasses: {self.class_names}\nCreated: {self.created_at}')
        if self.eval_results:
            for k, v in self.eval_results.items():
                print(f'  {k}: P={v["precision"]:.3f} R={v["recall"]:.3f} F1={v["f1"]:.3f}')

pipeline = ModelPipeline(
    model_path=best_pt,
    class_names=dict(enumerate(class_names)),
    training_config={'epochs': 100, 'batch': 16, 'device': 'cuda', 'imgsz': 640},
    eval_results=eval_results or None,
)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
pipeline.save(str(RUNS_DIR / 'pipeline.pkl'))
pipeline.summary()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/JetsonNano-Testing/DataTraining/classes.txt'